### Bước 1: Kết nối Google Drive

In [4]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Tạo thư mục lưu trữ Dataset trong Drive của bạn để quản lý gọn gàng
save_path = '/content/drive/MyDrive/datasets'
os.makedirs(save_path, exist_ok=True)
print(f"Thư mục lưu trữ đã sẵn sàng tại: {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thư mục lưu trữ đã sẵn sàng tại: /content/drive/MyDrive/datasets


### Bước 2: Tải Dataset vào Google Drive (Part 1 & 2)
Chúng ta sẽ thay đổi thư mục làm việc hiện tại (`os.chdir`) sang thư mục Drive trước khi tải để Roboflow lưu trực tiếp vào đó.

In [5]:
# Chuyển hướng thư mục làm việc sang Drive
os.chdir(save_path)

# Tải Part 1
from roboflow import Roboflow
rf1 = Roboflow(api_key=__import__("os").environ["ROBOFLOW_API_KEY"])
project1 = rf1.workspace("chikiens-workspace").project("1-prj-person-detection-dataset")
version1 = project1.version(2)
dataset1 = version1.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to 1-prj-person-detection-dataset-2 in yolov11:: 100%|██████████| 2005/2005 [00:15<00:00, 127.48it/s]


In [6]:
# Tải Part 2
rf2 = Roboflow(api_key=__import__("os").environ["ROBOFLOW_API_KEY"])
project2 = rf2.workspace("chikiens-workspace").project("2-prj-person-detection-dataset")
version2 = project2.version(1)
dataset2 = version2.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to 2-prj-person-detection-dataset-1 in yolov11:: 100%|██████████| 2003/2003 [00:17<00:00, 115.98it/s]


In [9]:
rf = Roboflow(api_key=__import__("os").environ["ROBOFLOW_API_KEY"])
project = rf.workspace("chikiens-workspace").project("3-prj-person-detection-dataset")
version = project.version(1)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to 3-prj-person-detection-dataset-1 in yolov11:: 100%|██████████| 2005/2005 [00:16<00:00, 123.66it/s]


In [10]:

from roboflow import Roboflow
rf = Roboflow(api_key=__import__("os").environ["ROBOFLOW_API_KEY"])
project = rf.workspace("chikiens-workspace").project("4-prj-person-detection-dataset")
version = project.version(1)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to 4-prj-person-detection-dataset-1 in yolov11:: 100%|██████████| 1285/1285 [00:10<00:00, 123.89it/s]


### Bước 3: Gộp 4 bộ Dataset (train, valid, test) lại với nhau trong Google Drive

In [12]:
import os
import shutil
from tqdm import tqdm

# Đường dẫn gốc chứa 4 datasets trong Drive
base_path = '/content/drive/MyDrive/datasets'

# Danh sách 4 thư mục dataset nguồn của bạn
source_datasets = [
    '1-prj-person-detection-dataset-2',
    '2-prj-person-detection-dataset-1',
    '3-prj-person-detection-dataset-1',
    '4-prj-person-detection-dataset-1'
]

splits = ['train', 'valid', 'test']
sub_dirs = ['images', 'labels']

# Tạo cấu trúc thư mục gộp trực tiếp tại base_path
for split in splits:
    for sub_dir in sub_dirs:
        os.makedirs(os.path.join(base_path, split, sub_dir), exist_ok=True)

print("=== Bắt đầu tiến trình gộp dữ liệu ===")

for idx, ds_name in enumerate(source_datasets):
    ds_src_path = os.path.join(base_path, ds_name)
    print(f"\nĐang xử lý {ds_name}...")

    if not os.path.exists(ds_src_path):
        print(f"⚠️ Không tìm thấy thư mục {ds_name}, bỏ qua.")
        continue

    for split in splits:
        src_split_path = os.path.join(ds_src_path, split)
        if not os.path.exists(src_split_path):
            continue

        for sub_dir in sub_dirs:
            src_dir = os.path.join(src_split_path, sub_dir)
            dst_dir = os.path.join(base_path, split, sub_dir)

            if not os.path.exists(src_dir):
                continue

            files = os.listdir(src_dir)
            for file_name in tqdm(files, desc=f"Gộp {split}/{sub_dir}"):
                src_file_path = os.path.join(src_dir, file_name)

                # Đổi tên file để tránh trùng lặp ghi đè giữa các dataset khác nhau
                new_file_name = f"ds{idx+1}_{file_name}"
                dst_file_path = os.path.join(dst_dir, new_file_name)

                # Copy file sang thư mục đích
                shutil.copy2(src_file_path, dst_file_path)

print("\n✅ Đã gộp thành công cả 4 datasets trực tiếp vào: /content/drive/MyDrive/datasets/")

=== Bắt đầu tiến trình gộp dữ liệu ===

Đang xử lý 1-prj-person-detection-dataset-2...


Gộp test/labels: 100%|██████████| 100/100 [00:01<00:00, 74.57it/s]



Đang xử lý 2-prj-person-detection-dataset-1...


Gộp test/labels: 100%|██████████| 100/100 [00:01<00:00, 79.42it/s]



Đang xử lý 3-prj-person-detection-dataset-1...


Gộp test/labels: 100%|██████████| 100/100 [00:01<00:00, 79.48it/s]



Đang xử lý 4-prj-person-detection-dataset-1...


Gộp test/labels: 100%|██████████| 64/64 [00:00<00:00, 72.60it/s]


✅ Đã gộp thành công cả 4 datasets trực tiếp vào: /content/drive/MyDrive/datasets/


### Tạo tệp cấu hình `data.yaml` mới cho Dataset đã gộp

In [13]:
import yaml

base_path = '/content/drive/MyDrive/datasets'

yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': {
        0: 'normal_person',
        1: 'potentially_dangerous_person'
    }
}

yaml_file_path = os.path.join(base_path, 'data.yaml')
with open(yaml_file_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print(f"📝 Đã tạo file cấu hình mới tại: {yaml_file_path}")

📝 Đã tạo file cấu hình mới tại: /content/drive/MyDrive/datasets/data.yaml


In [14]:
import os

base_path = '/content/drive/MyDrive/datasets'
splits = ['train', 'valid', 'test']
sub_dirs = ['images', 'labels']

print("=== THỐNG KÊ SỐ LƯỢNG FILE SAU KHI GỘP ===")
for split in splits:
    split_path = os.path.join(base_path, split)
    if os.path.exists(split_path):
        print(f"\nThư mục: {split}/")
        for sub_dir in sub_dirs:
            path = os.path.join(split_path, sub_dir)
            if os.path.exists(path):
                num_files = len(os.listdir(path))
                print(f"  - Thư mục con '{sub_dir}': {num_files} tệp tin")
            else:
                print(f"  - ⚠️ Không tìm thấy thư mục con: {sub_dir}")
    else:
        print(f"\n⚠️ Không tìm thấy thư mục: {split}/")

# Kiểm tra file data.yaml
yaml_path = os.path.join(base_path, 'data.yaml')
if os.path.exists(yaml_path):
    print(f"\n📝 File cấu hình 'data.yaml' tồn tại tại: {yaml_path}")
    print("--- Nội dung file data.yaml ---")
    with open(yaml_path, 'r') as f:
        print(f.read())
else:
    print(f"\n⚠️ Không tìm thấy file data.yaml tại: {base_path}")

=== THỐNG KÊ SỐ LƯỢNG FILE SAU KHI GỘP ===

Thư mục: train/
  - Thư mục con 'images': 2911 tệp tin
  - Thư mục con 'labels': 2911 tệp tin

Thư mục: valid/
  - Thư mục con 'images': 364 tệp tin
  - Thư mục con 'labels': 364 tệp tin

Thư mục: test/
  - Thư mục con 'images': 364 tệp tin
  - Thư mục con 'labels': 364 tệp tin

📝 File cấu hình 'data.yaml' tồn tại tại: /content/drive/MyDrive/datasets/data.yaml
--- Nội dung file data.yaml ---
names:
  0: normal_person
  1: potentially_dangerous_person
path: /content/drive/MyDrive/datasets
test: test/images
train: train/images
val: valid/images



### Bước 4: Nén và tải Dataset về máy tính cá nhân (Download Dataset)
Chúng ta sẽ nén toàn bộ thư mục dữ liệu đã gộp và tệp cấu hình `data.yaml` thành một file `.zip` để dễ dàng tải về máy.

In [15]:
import shutil
import os

# Đường dẫn chứa dataset đã gộp trực tiếp
dataset_path = '/content/drive/MyDrive/datasets'
zip_output_path = '/content/merged_dataset'

print("📦 Đang nén thư mục dataset thành file ZIP (Quá trình này có thể mất vài phút)...")
# Tiến hành nén thư mục /content/drive/MyDrive/datasets thành tệp merged_dataset.zip
shutil.make_archive(zip_output_path, 'zip', dataset_path)

print(f"✅ Đã nén thành công! Tệp tin ZIP được lưu tại: {zip_output_path}.zip")
print(f"Kích thước tệp: {os.path.getsize(zip_output_path + '.zip') / (1024*1024):.2f} MB")

📦 Đang nén thư mục dataset thành file ZIP (Quá trình này có thể mất vài phút)...
✅ Đã nén thành công! Tệp tin ZIP được lưu tại: /content/merged_dataset.zip
Kích thước tệp: 190.03 MB


In [16]:
from google.colab import files

# Tiến hành tải file ZIP về máy tính của bạn
print("📥 Đang bắt đầu tải tệp tin về máy tính...")
files.download('/content/merged_dataset.zip')

📥 Đang bắt đầu tải tệp tin về máy tính...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Bước 5: Dọn dẹp Google Drive (Xóa các thư mục dataset gốc sau khi gộp)
Dưới đây là script để xóa 4 thư mục con Roboflow ban đầu nhằm giải phóng dung lượng Drive của bạn.

In [17]:
import shutil
import os

base_path = '/content/drive/MyDrive/datasets'
folders_to_delete = [
    '1-prj-person-detection-dataset-2',
    '2-prj-person-detection-dataset-1',
    '3-prj-person-detection-dataset-1',
    '4-prj-person-detection-dataset-1'
]

print("🧹 Bắt đầu tiến trình xóa dọn dẹp các thư mục gốc...")

for folder in folders_to_delete:
    folder_path = os.path.join(base_path, folder)
    if os.path.exists(folder_path):
        try:
            shutil.rmtree(folder_path)
            print(f"🗑️ Đã xóa thành công: {folder}")
        except Exception as e:
            print(f"❌ Lỗi khi xóa {folder}: {e}")
    else:
        print(f"ℹ️ Thư mục không tồn tại hoặc đã bị xóa trước đó: {folder}")

print("\n✨ Quá trình dọn dẹp hoàn tất!")

🧹 Bắt đầu tiến trình xóa dọn dẹp các thư mục gốc...
🗑️ Đã xóa thành công: 1-prj-person-detection-dataset-2
🗑️ Đã xóa thành công: 2-prj-person-detection-dataset-1
🗑️ Đã xóa thành công: 3-prj-person-detection-dataset-1
🗑️ Đã xóa thành công: 4-prj-person-detection-dataset-1

✨ Quá trình dọn dẹp hoàn tất!
